# Session 14 · Special Field Types, Generated Fields and Parameters in Spark

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Hamnakhan123/Diploma-in-Data-Science/blob/main/term-2/02-big-data-systems-with-hadoop/code/session-14-spark-field-types-and-parameters.ipynb)

Diploma in Data Science · Prepared by **Hamna Zahid Khan**, Junior Lecturer

Run the cells from top to bottom. Every example is self-contained, so you can also run any single cell on its own.

**Setup:** the Spark examples need PySpark (and Java, which Google Colab already has).

In [1]:
%pip install -q pyspark

### Example 1 · Dates, timestamps, time zones and decimals

In [2]:
from decimal import Decimal
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import StructType, StructField, StringType, DecimalType
spark = (SparkSession.builder.master("local[2]").appName("types")
         .config("spark.sql.session.timeZone", "Asia/Karachi")
         .config("spark.ui.showConsoleProgress", "false").getOrCreate())
spark.sparkContext.setLogLevel("ERROR")
spark.conf.set("spark.sql.session.timeZone", "Asia/Karachi")

schema = StructType([StructField("order_id", StringType()), StructField("placed", StringType()),
                     StructField("price", DecimalType(10, 2)), StructField("phone", StringType())])
df = spark.createDataFrame([("A1", "24/10/2026 21:15", Decimal("0.10"), "03001234567"),
                            ("A2", "01/11/2026 09:05", Decimal("0.20"), "03217654321")], schema)
typed = (df.withColumn("placed_ts", F.to_timestamp("placed", "dd/MM/yyyy HH:mm"))
           .withColumn("placed_date", F.to_date("placed_ts"))
           .withColumn("weekday", F.date_format("placed_ts", "EEEE"))
           .withColumn("utc", F.to_utc_timestamp("placed_ts", "Asia/Karachi"))
           .withColumn("days_to_eom", F.datediff(F.last_day("placed_date"), F.col("placed_date"))))
typed.select("order_id", "placed_ts", "utc", "weekday", "days_to_eom", "phone").show(truncate=False)
typed.printSchema()

print("decimal sum :", df.agg(F.sum("price")).first()[0])
print("double sum  :", 0.1 + 0.2)

+--------+-------------------+-------------------+--------+-----------+-----------+
|order_id|placed_ts          |utc                |weekday |days_to_eom|phone      |
+--------+-------------------+-------------------+--------+-----------+-----------+
|A1      |2026-10-24 21:15:00|2026-10-24 16:15:00|Saturday|7          |03001234567|
|A2      |2026-11-01 09:05:00|2026-11-01 04:05:00|Sunday  |29         |03217654321|
+--------+-------------------+-------------------+--------+-----------+-----------+

root
 |-- order_id: string (nullable = true)
 |-- placed: string (nullable = true)
 |-- price: decimal(10,2) (nullable = true)
 |-- phone: string (nullable = true)
 |-- placed_ts: timestamp (nullable = true)
 |-- placed_date: date (nullable = true)
 |-- weekday: string (nullable = true)
 |-- utc: timestamp (nullable = true)
 |-- days_to_eom: integer (nullable = true)



decimal sum : 0.30
double sum  : 0.30000000000000004


### Example 2 · Nested JSON: structs, arrays and maps

In [3]:
from pyspark.sql import SparkSession, functions as F
spark = (SparkSession.builder.master("local[2]").appName("json")
         .config("spark.ui.showConsoleProgress", "false").getOrCreate())
spark.sparkContext.setLogLevel("ERROR")

events = spark.createDataFrame([
  ('{"user":{"id":7,"city":"Karachi"},"items":[{"sku":"BK-1","qty":2,"price":450.0},{"sku":"PN-9","qty":1,"price":120.0}],"tags":{"channel":"app","promo":"SALE1111"}}',),
  ('{"user":{"id":9,"city":"Lahore"},"items":[{"sku":"BK-1","qty":1,"price":450.0}],"tags":{"channel":"web"}}',)], ["raw"])

schema = ("user STRUCT<id: INT, city: STRING>, "
          "items ARRAY<STRUCT<sku: STRING, qty: INT, price: DOUBLE>>, tags MAP<STRING, STRING>")
parsed = events.select(F.from_json("raw", schema).alias("e")).select("e.*")
parsed.printSchema()

parsed.select(F.col("user.id").alias("user_id"), "user.city",
              F.size("items").alias("n_items"),
              F.aggregate("items", F.lit(0.0), lambda acc, x: acc + x.qty * x.price).alias("basket_value"),
              F.element_at("tags", "channel").alias("channel"),
              F.coalesce(F.element_at("tags", "promo"), F.lit("none")).alias("promo")).show()

parsed.select(F.col("user.id").alias("user_id"), F.explode("items").alias("item")) \
      .select("user_id", "item.sku", "item.qty", "item.price").show()

root
 |-- user: struct (nullable = true)
 |    |-- id: integer (nullable = true)
 |    |-- city: string (nullable = true)
 |-- items: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- sku: string (nullable = true)
 |    |    |-- qty: integer (nullable = true)
 |    |    |-- price: double (nullable = true)
 |-- tags: map (nullable = true)
 |    |-- key: string
 |    |-- value: string (valueContainsNull = true)



+-------+-------+-------+------------+-------+--------+
|user_id|   city|n_items|basket_value|channel|   promo|
+-------+-------+-------+------------+-------+--------+
|      7|Karachi|      2|      1020.0|    app|SALE1111|
|      9| Lahore|      1|       450.0|    web|    none|
+-------+-------+-------+------------+-------+--------+



+-------+----+---+-----+
|user_id| sku|qty|price|
+-------+----+---+-----+
|      7|BK-1|  2|450.0|
|      7|PN-9|  1|120.0|
|      9|BK-1|  1|450.0|
+-------+----+---+-----+



### Example 3 · Generated fields

In [4]:
from pyspark.sql import SparkSession, functions as F, Window
from pyspark.sql.functions import pandas_udf
import pandas as pd
spark = (SparkSession.builder.master("local[2]").appName("generated")
         .config("spark.sql.shuffle.partitions", "2")
         .config("spark.ui.showConsoleProgress", "false").getOrCreate())
spark.sparkContext.setLogLevel("ERROR")

sales = spark.createDataFrame([("Karachi", "Ayesha", 5200.0), ("Lahore", "Bilal", 900.0),
                               ("Karachi", "Omar", 15500.0), ("Lahore", "Sana", 7300.0),
                               ("Karachi", "Hina", 2100.0)], "city string, customer string, amount double")

@pandas_udf("string")
def initials(names: pd.Series) -> pd.Series:          # vectorised UDF: runs on whole batches
    return names.str[0] + "."

w_city = Window.partitionBy("city")
gen = (sales
   .withColumn("row_id", F.row_number().over(Window.orderBy("customer")))
   .withColumn("tier", F.when(F.col("amount") >= 10000, "Gold").when(F.col("amount") >= 5000, "Silver").otherwise("Bronze"))
   .withColumn("gst", F.round(F.col("amount") * 0.18, 2))
   .withColumn("share_of_city_%", F.round(F.col("amount") / F.sum("amount").over(w_city) * 100, 1))
   .withColumn("rank_in_city", F.rank().over(w_city.orderBy(F.desc("amount"))))
   .withColumn("cust_key", F.substring(F.sha2(F.col("customer"), 256), 1, 10))
   .withColumn("anon", initials("customer")))
gen.orderBy("row_id").show()

+-------+--------+-------+------+------+------+---------------+------------+----------+----+
|   city|customer| amount|row_id|  tier|   gst|share_of_city_%|rank_in_city|  cust_key|anon|
+-------+--------+-------+------+------+------+---------------+------------+----------+----+
|Karachi|  Ayesha| 5200.0|     1|Silver| 936.0|           22.8|           2|8d3cf93037|  A.|
| Lahore|   Bilal|  900.0|     2|Bronze| 162.0|           11.0|           2|dd5f1c0842|  B.|
|Karachi|    Hina| 2100.0|     3|Bronze| 378.0|            9.2|           3|14305a77ce|  H.|
|Karachi|    Omar|15500.0|     4|  Gold|2790.0|           68.0|           1|2dd1283eb2|  O.|
| Lahore|    Sana| 7300.0|     5|Silver|1314.0|           89.0|           1|1b1902423d|  S.|
+-------+--------+-------+------+------+------+---------------+------------+----------+----+



### Example 4 · Parameterised SQL, broadcast variables and accumulators

In [5]:
import argparse
from pyspark.sql import SparkSession, functions as F
spark = (SparkSession.builder.master("local[2]").appName("params")
         .config("spark.ui.showConsoleProgress", "false").getOrCreate())
spark.sparkContext.setLogLevel("ERROR")
sc = spark.sparkContext

# 1) job arguments (in production: spark-submit job.py --city Karachi --min-amount 3000)
parser = argparse.ArgumentParser()
parser.add_argument("--city", default="Karachi")
parser.add_argument("--min-amount", type=float, default=3000)
args = parser.parse_args([])                                   # [] so it also runs in a notebook

spark.createDataFrame([(1, "Karachi", 5200.0), (2, "Lahore", 900.0), (3, "Karachi", 15500.0),
                       (4, "Lahore", 7300.0), (5, "Karachi", 2100.0)],
                      "id int, city string, amount double").createOrReplaceTempView("sales")

# 2) named parameter markers: values are bound, never pasted into the SQL text
q = "SELECT id, city, amount FROM sales WHERE city = :city AND amount >= :min_amt ORDER BY id"
spark.sql(q, args={"city": args.city, "min_amt": args.min_amount}).show()

# 3) IDENTIFIER() for a table name chosen at run time
print("rows:", spark.sql("SELECT COUNT(*) AS n FROM IDENTIFIER(:tbl)", args={"tbl": "sales"}).first()["n"])

# 4) a malicious value is treated as data, not SQL
evil = "Karachi' OR '1'='1"
print("injection attempt rows:", spark.sql("SELECT * FROM sales WHERE city = :city", args={"city": evil}).count())

# 5) broadcast variable + accumulator on an RDD
tax = sc.broadcast({"Karachi": 0.13, "Lahore": 0.16})          # shipped once per executor
bad = sc.accumulator(0)
def add_tax(r):
    rate = tax.value.get(r[1])
    if rate is None:
        bad.add(1); return None
    return (r[0], r[1], round(r[2] * (1 + rate), 1))
rows = sc.parallelize([(1, "Karachi", 1000.0), (2, "Lahore", 1000.0), (3, "Quetta", 1000.0)])
print("with tax:", rows.map(add_tax).filter(lambda x: x).collect(), "| unknown cities:", bad.value)

+---+-------+-------+
| id|   city| amount|
+---+-------+-------+
|  1|Karachi| 5200.0|
|  3|Karachi|15500.0|
+---+-------+-------+



rows: 5


injection attempt rows: 0


with tax: [(1, 'Karachi', 1130.0), (2, 'Lahore', 1160.0)] | unknown cities: 1
